In [1]:
# --- STEP 1: INSTALL AND IMPORT NECESSARY LIBRARIES ---
# kagglehub is used to download the dataset directly from Kaggle
!pip install kagglehub

import os                       # For interacting with the operating system/file paths
import cv2                      # OpenCV for image processing and resizing
import joblib                   # To save the trained model to a file
import numpy as np              # For numerical operations and array handling
import kagglehub                # The tool you provided for direct dataset download
from skimage.feature import hog # To extract Histogram of Oriented Gradients (HOG) features
from sklearn.ensemble import RandomForestClassifier # The Classical ML algorithm
from sklearn.model_selection import train_test_split # To split data into Train/Test sets
from sklearn.metrics import accuracy_score, classification_report # For evaluation

In [2]:
# SEGMENT 2: DATA ACQUISITION & PATH FINDING
# ==========================================
print("Step 1: Downloading RTK dataset...")
# Downloads the Road Traversing Knowledge dataset [cite: 18]
base_path = kagglehub.dataset_download("tallwinkingstan/road-traversing-knowledge-rtk-dataset")

# Define our target road categories
categories = ['Asphalt', 'Paved', 'Unpaved']
data = []
labels = []

print("Step 2: Scanning for image folders...")

Step 1: Downloading RTK dataset...


100%|██████████| 723M/723M [00:47<00:00, 15.9MB/s]

Extracting files...


Step 2: Scanning for image folders...


In [3]:
# ==========================================
# SEGMENT 3: PREPROCESSING & FEATURE EXTRACTION
# ==========================================
def get_hog_features(img_path):
    """
    Handles preprocessing (resizing/grayscale) and feature extraction (HOG).
    This fulfills the requirement for Data Preprocessing[cite: 20].
    """
    img = cv2.imread(img_path)
    # Resize to 128x128 to standardize the input size [cite: 20]
    img = cv2.resize(img, (128, 128))
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)

    # Extract HOG features to capture texture and road structure [cite: 23]
    features = hog(gray, orientations=9, pixels_per_cell=(16, 16), cells_per_block=(2, 2))
    return features

In [4]:
# ==========================================
# SEGMENT 4: DATA LOADING LOOP
# ==========================================
# The RTK dataset is split into parts; this loop aggregates them into our 3 categories.
for root, dirs, files in os.walk(base_path):
    for directory in dirs:
        dir_lower = directory.lower()

        # Map folder names to our numeric labels
        current_label = -1
        if 'asphalt' in dir_lower: current_label = 0
        elif 'paved' in dir_lower: current_label = 1
        elif 'unpaved' in dir_lower: current_label = 2

        if current_label != -1:
            folder_path = os.path.join(root, directory)
            images = [f for f in os.listdir(folder_path) if f.lower().endswith(('.png', '.jpg', '.jpeg'))]

            if len(images) > 0:
                print(f"Found {len(images)} images in {directory} -> Mapping to {categories[current_label]}")
                # Process a sample of 50 images per part for an efficient baseline [cite: 25]
                for img_name in images[:50]:
                    try:
                        feat = get_hog_features(os.path.join(folder_path, img_name))
                        data.append(feat)
                        labels.append(current_label)
                    except:
                        continue

Found 124 images in pavedBad -> Mapping to Paved
Found 324 images in pavedRegular -> Mapping to Paved
Found 1978 images in asphaltGood -> Mapping to Asphalt
Found 464 images in asphaltBad -> Mapping to Asphalt
Found 839 images in asphaltRegular -> Mapping to Asphalt
Found 796 images in unpavedRegular -> Mapping to Paved
Found 593 images in unpavedBad -> Mapping to Paved


In [6]:
# ==========================================
# SEGMENT 5: TRAINING, EVALUATION, & SAVING
# ==========================================
if len(data) > 0:
    # 1. Split the data
    X_train, X_test, y_train, y_test = train_test_split(data, labels, test_size=0.2, random_state=42)

    # 2. Train the Random Forest
    print("\nStep 3: Training Random Forest (Classical ML)...")
    model = RandomForestClassifier(n_estimators=100, random_state=42)
    model.fit(X_train, y_train)

    # 3. Evaluate the model
    y_pred = model.predict(X_test)

    # --- FIX FOR VALUEERROR ---
    # Identify which unique classes were actually present in the test set
    unique_labels = np.unique(np.concatenate((y_test, y_pred)))
    actual_target_names = [categories[i] for i in unique_labels]

    print(f"\n✅ STAGE 1 SUCCESS!")
    print(f"Model Accuracy: {accuracy_score(y_test, y_pred) * 100:.2f}%")

    # The classification_report now only uses the names of the classes it found
    print("\nDetailed Performance Report:\n",
          classification_report(y_test, y_pred, target_names=actual_target_names))

    # 4. Save the model for your Streamlit GUI
    joblib.dump(model, 'rtk_classical_model.pkl')
    print("\nSaved: 'rtk_classical_model.pkl' is ready for the GUI.")

else:
    print("❌ Error: No images were found. Check the dataset path or structure.")


Step 3: Training Random Forest (Classical ML)...

✅ STAGE 1 SUCCESS!
Model Accuracy: 94.29%

Detailed Performance Report:
               precision    recall  f1-score   support

     Asphalt       0.93      0.93      0.93        29
       Paved       0.95      0.95      0.95        41

    accuracy                           0.94        70
   macro avg       0.94      0.94      0.94        70
weighted avg       0.94      0.94      0.94        70


Saved: 'rtk_classical_model.pkl' is ready for the GUI.
